In [5]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [6]:
df = pd.read_csv('C:/Sarang/College/lp3/ml/uber.csv')

In [7]:
print(f'\nShape: {df.shape}')
df.head()

In [8]:
df.isnull().sum()

In [9]:
df.duplicated().sum()

In [10]:
df.dropna()

In [11]:
df.drop(columns=["Unnamed: 0", "key"], inplace=True)
df.columns

In [12]:
df["pickup_datetime"] = pd.to_datetime(
    df["pickup_datetime"],
    errors="coerce"
)

In [13]:
df.dropna(inplace=True)

In [14]:
df.shape

In [15]:
df["year"] = df["pickup_datetime"].dt.year
df["month"] = df["pickup_datetime"].dt.month
df["day"] = df["pickup_datetime"].dt.day
df["hour"] = df["pickup_datetime"].dt.hour
df["weekday"] = df["pickup_datetime"].dt.dayofweek

In [16]:
df.drop(columns=["pickup_datetime"], inplace=True)
print(df.head())

In [17]:
df = df[df["fare_amount"] > 0]
df = df[
    (df["passenger_count"] > 0) &
    (df["passenger_count"] <= 6)
]

In [18]:
df = df[
    (df["pickup_longitude"].between(-75, -72)) &
    (df["dropoff_longitude"].between(-75, -72)) &
    (df["pickup_latitude"].between(40, 42)) &
    (df["dropoff_latitude"].between(40, 42))
]

In [19]:
print(df.shape)

In [20]:
def haversine_distance(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


df["distance_km"] = haversine_distance(
    df["pickup_latitude"],
    df["pickup_longitude"],
    df["dropoff_latitude"],
    df["dropoff_longitude"]
)

In [21]:
df["distance_km"].describe()

In [22]:
def find_outliers_iqr(data, column):

    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]

    print(f"\nColumn: {column}")
    print(f"Q1 = {Q1:.2f}")
    print(f"Q3 = {Q3:.2f}")
    print(f"IQR = {IQR:.2f}")
    print(f"Lower bound = {lower_bound:.2f}")
    print(f"Upper bound = {upper_bound:.2f}")
    print(f"Number of outliers = {len(outliers)}")

    return outliers

for column in [
    "fare_amount",
    "passenger_count",
    "distance_km"
]:
    find_outliers_iqr(df, column)

In [23]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.boxplot(y=df["fare_amount"])
plt.title("Fare Amount Outliers")

plt.subplot(1, 3, 2)
sns.boxplot(y=df["passenger_count"])
plt.title("Passenger Count Outliers")

plt.subplot(1, 3, 3)
sns.boxplot(y=df["distance_km"])
plt.title("Distance Outliers")

plt.tight_layout()
plt.show()

In [24]:
df = df[
    (df["fare_amount"] <= 100) &
    (df["distance_km"] <= 100)
]

df = df[df["distance_km"] >= 0]
print(df.shape)

In [25]:
correlation_matrix = df.corr()

print("\nCorrelation matrix:")
print(correlation_matrix["fare_amount"].sort_values(ascending=False))
plt.figure(figsize=(12, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [26]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df.sample(min(10000, len(df)), random_state=42),
    x="distance_km",
    y="fare_amount",
    alpha=0.4
)

plt.title("Fare Amount vs Distance")
plt.xlabel("Distance (km)")
plt.ylabel("Fare Amount ($)")
plt.show()

In [27]:
X = df.drop(columns=["fare_amount"])

y = df["fare_amount"]

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget:")
print("fare_amount")

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)

In [29]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)

In [30]:
random_forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

y_pred_rf = random_forest_model.predict(X_test)

In [31]:
def evaluate_model(name, y_true, y_pred):

    r2 = r2_score(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(y_true, y_pred)

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print(f"R² Score : {r2:.4f}")
    print(f"MSE      : {mse:.4f}")
    print(f"RMSE     : {rmse:.4f}")
    print(f"MAE      : {mae:.4f}")

    return r2, mse, rmse, mae


linear_scores = evaluate_model(
    "Linear Regression",
    y_test,
    y_pred_linear
)


rf_scores = evaluate_model(
    "Random Forest Regression",
    y_test,
    y_pred_rf
)

In [32]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest Regression"
    ],

    "R2": [
        linear_scores[0],
        rf_scores[0]
    ],

    "MSE": [
        linear_scores[1],
        rf_scores[1]
    ],

    "RMSE": [
        linear_scores[2],
        rf_scores[2]
    ],

    "MAE": [
        linear_scores[3],
        rf_scores[3]
    ]
})

print("\nModel Comparison:")
print(results)

In [33]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=results,
    x="Model",
    y="R2",
    hue="Model",
)

plt.title("R² Score Comparison")
plt.ylabel("R² Score")
plt.xticks(rotation=10)
plt.show()


plt.figure(figsize=(10, 5))

sns.barplot(
    data=results,
    x="Model",
    y="RMSE",
    hue="Model",
)

plt.title("RMSE Comparison")
plt.ylabel("RMSE")
plt.xticks(rotation=10)
plt.show()

In [34]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    y_pred_rf,
    alpha=0.3
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red",
    linestyle="--"
)

plt.xlabel("Actual Fare")
plt.ylabel("Predicted Fare")
plt.title("Random Forest: Actual vs Predicted Fare")

plt.show()

In [35]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": random_forest_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance:")
print(feature_importance)


plt.figure(figsize=(10, 6))

sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature"
)

plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

In [36]:
pickup_lat = 40.7580
pickup_lon = -73.9855

dropoff_lat = 40.7484
dropoff_lon = -73.9857

passenger_count = 2

pickup_datetime = pd.Timestamp("2026-09-08 18:30:00")

distance = haversine_distance(
    pickup_lat,
    pickup_lon,
    dropoff_lat,
    dropoff_lon
)

new_ride = pd.DataFrame({
    "pickup_longitude": [pickup_lon],
    "pickup_latitude": [pickup_lat],
    "dropoff_longitude": [dropoff_lon],
    "dropoff_latitude": [dropoff_lat],
    "passenger_count": [passenger_count],
    "year": [pickup_datetime.year],
    "month": [pickup_datetime.month],
    "day": [pickup_datetime.day],
    "hour": [pickup_datetime.hour],
    "weekday": [pickup_datetime.dayofweek],
    "distance_km": [distance]
})

new_ride = new_ride[X.columns]

predicted_fare = random_forest_model.predict(new_ride)[0]

print(f"Distance      : {distance:.2f} km")
print(f"Passengers    : {passenger_count}")
print(f"Predicted fare: ${predicted_fare:.2f}")